# Conexões com Spark

Este notebook mostra, na prática, as principais formas de conectar um cliente Python (PySpark) a um Apache Spark — da mais simples (um processo local, sem cluster nenhum) até conexões remotas com clusters reais rodando em Docker.

Vamos comparar três cenários:

- **Caso A — Spark local (Standalone)**: o próprio processo Python sobe um Spark "de bolso" na sua máquina, sem containers. Ideal para testes rápidos e para aprender a API.
- **Caso B — Spark Cluster**: o cliente Python se conecta a um cluster de verdade rodando em Docker (1 master + 2 workers), de duas formas — via **Spark Connect** (`sc://...`, o padrão recomendado hoje) e via conexão direta ao master (`spark://...`, o modo clássico).
- **Caso C — Spark Connect + HDFS**: o cliente Python se conecta a um Spark Connect que lê e escreve no **HDFS** (`hdfs://namenode:8020`), sem YARN — o Spark processa, o HDFS armazena.

Ao final, você vai entender a diferença entre "ter um Spark local" e "conectar a um Spark remoto" — e por que ela importa na hora de encerrar a sessão (`spark.stop()` nem sempre é inofensivo!).

# Caso A: Spark local Standalone

Aqui o Spark roda **dentro do próprio processo Python**, em modo *Standalone* local — sem master, sem workers, sem containers Docker envolvidos. É a forma mais simples de começar: basta ter o PySpark instalado.

Use esse modo para prototipar transformações, testar código rapidamente e aprender a API sem depender de nenhuma infraestrutura externa.

## Conexão simples

Criamos uma `SparkSession` com as configurações padrão do Apache Spark, apontando `.master("local[*]")`.

O `*` diz ao Spark para usar **todos os núcleos de CPU disponíveis na máquina** como se fossem "workers" — é assim que ele consegue paralelizar tarefas mesmo sem um cluster de verdade por trás.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("app-01")
    .master("local[*]")     # Em local[*] isso controla quantas tarefas rodam em paralelo
    .getOrCreate()
)
# Exibe a representação da SparkSession criada
spark
print(f"✅ Conexão ativa {spark}: Acessar http://localhost:4040")

In [ ]:
# Fechar conexão
spark.stop()

## Conexões customizadas

A `SparkSession` aceita configurações via `.config(chave, valor)` para ajustar como o Spark processa os dados. Alguns parâmetros comuns:

- **`spark.sql.shuffle.partitions`**: número de partições usadas em operações de *shuffle* (`groupBy`, `join`, etc.). O padrão é 200; em máquinas pequenas ou com datasets pequenos, um valor baixo (como 8) evita o overhead de criar centenas de partições praticamente vazias.
- **`spark.executor.memory`**: quanta memória cada executor pode usar para processar dados.
- **`spark.executor.cores`**: quantos núcleos de CPU cada executor pode usar em paralelo.

Calibrar esses valores é essencial para performance: partições de menos gera gargalo (pouco paralelismo); partições demais gera overhead de coordenação entre tarefas muito pequenas.

Para ver um conjunto completo de configurações visite: [https://spark.apache.org/docs/latest/configuration.html](https://spark.apache.org/docs/latest/configuration.html)

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("app-01")
    .master("local[*]") 
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.cores", "2")
    .getOrCreate()
)
# Exibe a representação da SparkSession criada
spark
print(f"✅ Conexão ativa {spark}: Acessar http://localhost:4040")

In [ ]:
# Fechar conexão
spark.stop()

# Caso B: Spark Cluster

Agora o Spark roda **fora do processo Python**, em um cluster de verdade formado por containers Docker: 1 nó *master* (coordenador) e 2 nós *workers* (que executam as tarefas de fato).

Suba o cluster antes de rodar as células abaixo:

```bash
make spark
```

Esse comando inicia o master, os workers e o serviço **Spark Connect** — a porta de entrada que os clientes Python usam para conversar com o cluster.

## Conexão simples

Em vez de `.master("local[*]")`, usamos `.remote("sc://localhost:15002")`.

- `.remote(...)` diz ao PySpark para falar o protocolo **Spark Connect** — uma API gRPC leve que permite conectar a um Spark remoto sem precisar instalar Spark/Hadoop localmente.
- `sc://localhost:15002` aponta para o serviço **Spark Connect**, que atua como *proxy* de entrada do cluster: recebe os comandos do cliente, traduz em jobs Spark e devolve os resultados.
- Diferença importante em relação ao Caso A: aqui o `SparkContext` **não pertence ao cliente Python** — ele vive dentro do container `spark-connect` e é compartilhado por qualquer cliente que se conecte nessa porta. Isso tem uma consequência direta na hora de encerrar a sessão — veja o aviso na célula logo abaixo antes de rodar `spark.stop()`.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("app-02")
    .remote("sc://localhost:15002")   # remote aponta a conexão para o serviço Spark Connect
    .getOrCreate()
)
# Exibe a representação da SparkSession criada
spark
print("🖥️  Master UI:    http://localhost:8080")
print("🖥️  Worker UIs:   http://localhost:8081  http://localhost:8082")
print("📊 Spark App UI:  http://localhost:4040")

### ⚠️ Por que não chamar `spark.stop()` aqui

No Caso A, cada `spark.stop()` encerra uma sessão **local e privada** — sem efeitos colaterais para mais ninguém.

No Caso B isso muda: o `SparkContext` é **compartilhado** por todo o cluster (vive dentro do container `spark-connect`). Chamar `spark.stop()` no cliente derruba esse `SparkContext` para **todo mundo que estiver conectado**, não apenas a sua sessão.

**Sintoma:** o container continua aparecendo como `Up` e `healthy` no `docker compose ps`, mas qualquer nova conexão falha com:

```
UserWarning: <_InactiveRpcError ... status = StatusCode.UNKNOWN>
```

Isso acontece porque o serviço tenta reaproveitar um `SparkContext` que já foi parado.

**Como recuperar o cluster** (no terminal):

```bash
docker compose restart spark-connect   # reinicia só o serviço Spark Connect
# ou
make down && make spark        # reinicia o cluster inteiro
```

**Como encerrar a sessão sem derrubar o cluster:** simplesmente não chame `spark.stop()` — a conexão gRPC é liberada automaticamente quando o kernel do notebook é reiniciado ou encerrado.

## Conexão direta ao master (modo cluster clássico)

Além do Spark Connect, também é possível conectar diretamente ao **Standalone Cluster Manager** do Spark, apontando para a URL do master:

- `.master("spark://localhost:7077")` — em vez de um proxy gRPC (Spark Connect), aqui o seu processo Python **é o próprio Driver** da aplicação Spark, e conversa diretamente com o `spark-master` para negociar executores nos workers.
- Essa é a forma "clássica" de conectar a um cluster Spark Standalone — o modelo usado antes do Spark Connect existir, e ainda muito comum em jobs submetidos via `spark-submit`.

⚠️ **Diferença de rede importante**: o Driver roda no seu host, mas os executores rodam dentro dos containers (`spark-worker-1`/`spark-worker-2`) — eles precisam conseguir se conectar de volta ao Driver. Isso exige **duas** configurações distintas, que são frequentemente confundidas:

- **`spark.driver.host`**: o endereço que os **executores** usam para alcançar o Driver de volta. Usamos `host.docker.internal`, o hostname que containers Docker resolvem para o host.
- **`spark.driver.bindAddress`**: o endereço no qual o **próprio Driver** abre seu socket local. Por padrão, o Spark usa o mesmo valor de `spark.driver.host` — mas `host.docker.internal` só é resolvível *de dentro* dos containers, não no seu host! Sem setar isso explicitamente para `0.0.0.0`, o Driver tenta dar bind num endereço que ele mesmo não consegue resolver, e quebra com `UnresolvedAddressException`.

⚠️ **Pegadinha do mesmo kernel**: ao criar a sessão Spark Connect da célula anterior, o PySpark marca `os.environ["SPARK_CONNECT_MODE_ENABLED"] = "1"` — e nunca limpa essa marca, nem com `spark.stop()`. Se essa variável continuar setada, a próxima chamada a `getOrCreate()` tenta reentrar no modo Connect e quebra com `AttributeError: 'NoneType' object has no attribute 'startswith'` (não existe `spark.remote` configurado dessa vez). Por isso a célula abaixo remove essa variável antes de criar a sessão clássica.

In [ ]:
import os
from pyspark.sql import SparkSession

# Limpa a marca deixada pela sessão Spark Connect anterior — sem isso, getOrCreate()
# tenta reentrar no modo Connect e quebra (ver aviso acima)
os.environ.pop("SPARK_CONNECT_MODE_ENABLED", None)

spark = (
    SparkSession.builder.appName("app-03")
    .master("spark://localhost:7077")  # conecta direto no Standalone Cluster Manager
    .config("spark.driver.host", "host.docker.internal")  # workers alcançam o Driver por aqui
    .config("spark.driver.bindAddress", "0.0.0.0")  # o Driver faz bind localmente em todas as interfaces
    .getOrCreate()
)
# Exibe a representação da SparkSession criada
spark
print(f"✅ Conexão ativa {spark}")
print("🖥️  Master UI: http://localhost:8080")

### ✅ Aqui `spark.stop()` é seguro

Diferente da conexão via Spark Connect, nesse modo o seu processo Python é o dono da aplicação registrada no master — confira em http://localhost:8080 e você verá uma nova *Application* chamada `app-03`. Chamar `spark.stop()` encerra **apenas essa aplicação**, sem afetar o `spark-connect` nem qualquer outro cliente conectado ao cluster.

In [ ]:
# Fechar conexão — seguro: encerra só a aplicação "app-03", não o cluster inteiro
spark.stop()

# Caso C: Spark Connect + HDFS

Por fim, o Spark também pode ler e escrever no **HDFS** (Hadoop Distributed File System). Diferente do YARN (que gerencia recursos), o HDFS é apenas um sistema de arquivos distribuído — o Spark precisa de um gerenciador de cluster separado para alocar os executores.

A arquitetura do Caso C usa **Spark Connect** contra um cluster Spark Standalone que tem o HDFS como sistema de arquivos nativo (`hdfs://namenode:8020`). O cliente Python se conecta via `sc://localhost:15004` (Spark Connect) e usa os DataFrames apontando para `hdfs://` — exatamente a mesma API do Caso D com S3, mas contra um filesystem.

⚠️ **Este caso é apenas ilustrativo aqui.** Executá-lo exige infraestrutura extra rodando (`make hadoop`) e os dados precisam ser carregados no HDFS via gateway HttpFS (`http://localhost:14000`) antes de serem processados. O código abaixo mostra a conexão real, mas como texto.

## Conexão via Spark Connect

- `.remote("sc://localhost:15004")` conecta ao servidor **Spark Connect** que roda dentro do profile hadoop. Os executores (spark-worker-1, spark-worker-2) enxergam o HDFS nativamente via `hdfs://namenode:8020`.
- O cliente Python não precisa de configurações Hadoop locais — todo o acesso ao HDFS é resolvido do lado do servidor Spark.
- Os dados brutos precisam estar no HDFS antes: o script `scripts/lab_utils.py` oferece `upload_bronze_table_to_hdfs()` que usa a API WebHDFS pública (http://localhost:14000) para copiar CSVs do host para o HDFS.
- A leitura/escrita de DataFrames usa `hdfs://namenode:8020/datalake/{camada}/...` como path — exatamente a mesma API DataFrame, mudando só o prefixo do filesystem.

```python
from pyspark.sql import SparkSession

spark = SparkSession.builder
    .remote("sc://localhost:15004")
    .appName("app-04")
    .getOrCreate()

df = spark.read.parquet("hdfs://namenode:8020/datalake/silver/vendas/")
df.show(5)
print("🖥️  NameNode UI: http://localhost:9870")
print("🖥️  Spark UI:    http://localhost:18080")
```